In [ ]:
import os
import rasterio
import numpy as np

# 입력 및 출력 경로 설정
input_folder = r"E:\projects\smelter\data\python\BlastFurnaces\TAI\0_1"
output_folder = os.path.join(input_folder, "sum")
os.makedirs(output_folder, exist_ok=True)
output_path = os.path.join(output_folder, "TAIsum.tif")

# 'S2'로 시작하는 TIF 파일만 목록에 추가
tif_files = [f for f in os.listdir(input_folder) if f.startswith('S2') and f.endswith('.tif')]

# 초기화
sum_array = None
meta = None

# 합산 처리
for i, tif_file in enumerate(tif_files):
    file_path = os.path.join(input_folder, tif_file)
    with rasterio.open(file_path) as src:
        data = src.read(1).astype(np.float32)
        
        # nodata 값을 0으로 처리
        if src.nodata is not None:
            data[data == src.nodata] = 0
        
        # NaN 값도 0으로 처리
        data[~np.isfinite(data)] = 0

        # 첫 파일이면 초기화
        if sum_array is None:
            sum_array = np.zeros_like(data, dtype=np.float32)
            meta = src.meta.copy()
        
        sum_array += data

# 결과 저장
meta.update(dtype=rasterio.float32, nodata=0)
with rasterio.open(output_path, 'w', **meta) as dst:
    dst.write(sum_array, 1)

print(f"✅ 저장 완료: {output_path}")


In [ ]:
############# TAI 합산 값이 지정한 값 이상일 때의 픽셀 좌표 #############

import os
import rasterio
import geopandas as gpd
import numpy as np
import pandas as pd
from shapely.geometry import Point

# 경로 설정
tif_path = r"E:\projects\smelter\data\python\BlastFurnaces\TAI\0_1\sum\TAIsum.tif"
shp_path = r"E:\projects\smelter\shp\BlastFurnaces_NoTorpedo.shp"
output_csv = r"E:\projects\smelter\data\python\BlastFurnaces\TAI\0_1\sum\TAIsum_150selected_pixels_nontope.csv"

# TIF 파일 열기
with rasterio.open(tif_path) as src:
    data = src.read(1)
    transform = src.transform
    crs = src.crs

# Shapefile 열기
gdf = gpd.read_file(shp_path)
gdf = gdf.to_crs(crs)  # TIF와 좌표계 통일

# 100 이상인 픽셀 찾기
mask = data >= 150
rows, cols = np.where(mask)

# 결과 저장용 리스트
selected_pixels = []

# 각 픽셀 좌표를 검사
for row, col in zip(rows, cols):
    x, y = rasterio.transform.xy(transform, row, col, offset='center')
    point = Point(x, y)
    if gdf.contains(point).any():
        selected_pixels.append((row, col))

# 결과 저장
df = pd.DataFrame(selected_pixels, columns=["Row", "Col"])
df.to_csv(output_csv, index=False, encoding="utf-8-sig")

print(f"✅ 완료: {output_csv}에 {len(df)}개 픽셀 저장됨")


In [ ]:
############# 날짜별 TAI 합산 값이 지정한 값 이상일 때 픽셀 좌표에서의 상관계수 #############

import os
import pandas as pd
import numpy as np
import rasterio
from scipy.stats import pearsonr, spearmanr
from tqdm import tqdm

# 경로 설정
base_folder = r"E:\projects\smelter\data\python\BlastFurnaces\TAI\0_1"
sum_folder = os.path.join(base_folder, "sum")
tif_folder = base_folder  # TIF 파일들은 여기
matched_tif_csv = os.path.join(base_folder, "matched_tif.csv")
selected_pixels_csv = os.path.join(sum_folder, "TAIsum_150selected_pixels_nontope.csv")
activity_csv = r"E:\projects\smelter\data\python\BlastFurnaces\Dillinger_activity_and_production.csv"

# 출력 경로
output_csv = os.path.join(sum_folder, "nTAIpositive_150vs_activity_nontope.csv")

# 읽기
df_matched = pd.read_csv(matched_tif_csv)
tif_files = df_matched['filename'].tolist()

df_pixels = pd.read_csv(selected_pixels_csv)
df_pixels.columns = df_pixels.columns.str.lower()
pixel_coords = list(zip(df_pixels['row'], df_pixels['col']))

df_activity = pd.read_csv(activity_csv)
df_activity["Date"] = pd.to_datetime(df_activity["Capture time"])

# 날짜별 n_TAI_positive 계산
records = []

for tif_file in tqdm(tif_files, desc="🔍 날짜별 0 초과 픽셀 개수 계산 중"):
    tif_path = os.path.join(tif_folder, tif_file)
    try:
        with rasterio.open(tif_path) as src:
            img = src.read(1)

            # 관심 있는 픽셀들의 값만 가져옴
            pixel_values = np.array([img[row, col] for row, col in pixel_coords])

            # 0 초과하는 값 세기
            n_positive = np.sum(pixel_values > 0)

        # 파일명에서 날짜 추출
        parts = tif_file.split('_')
        if len(parts) > 2:
            date_str = parts[2][:8]  # YYYYMMDD
            date = pd.to_datetime(date_str, format="%Y%m%d")
        else:
            continue

        # 해당 날짜에 대응하는 activity 찾기
        activity_row = df_activity[df_activity["Date"] == date]
        if not activity_row.empty:
            activity = activity_row["Activity"].values[0]
            records.append((date, n_positive, activity))

    except Exception as e:
        print(f"⚠️ 오류 발생: {tif_file} - {e}")

# DataFrame 생성
df_result = pd.DataFrame(records, columns=["Date", "n_TAI_positive", "Activity"])
df_result.sort_values("Date", inplace=True)

# 저장
df_result.to_csv(output_csv, index=False, encoding="utf-8-sig")
print(f"✅ 결과 저장 완료: {output_csv}")

# 상관계수 계산
r_pearson, _ = pearsonr(df_result["n_TAI_positive"], df_result["Activity"])
r_spearman, _ = spearmanr(df_result["n_TAI_positive"], df_result["Activity"])

print(f"\n📈 Pearson 상관계수 (n_TAI_positive vs Activity): {r_pearson:.4f}")
print(f"📈 Spearman 상관계수 (n_TAI_positive vs Activity): {r_spearman:.4f}")


In [ ]:
############# 상관계수 값이 지정한 값 이상일 때의 픽셀 좌표 #############

import os
import rasterio
import geopandas as gpd
import numpy as np
import pandas as pd
from shapely.geometry import Point

# 경로 설정
base_folder = r"E:\projects\smelter\data\python\BlastFurnaces\TAI\0_1"
pearson_path = os.path.join(base_folder, "correlation_map_pearson.tif")
spearman_path = os.path.join(base_folder, "correlation_map_spearman.tif")
shp_path = r"E:\projects\smelter\shp\Blast Furnaces.shp"
output_csv = os.path.join(base_folder, "correlation_01selected_pixels.csv")

# TIF 파일 열기
with rasterio.open(pearson_path) as src_pearson, rasterio.open(spearman_path) as src_spearman:
    pearson_data = src_pearson.read(1)
    spearman_data = src_spearman.read(1)
    transform = src_pearson.transform
    crs = src_pearson.crs

# Shapefile 열기
gdf = gpd.read_file(shp_path)
gdf = gdf.to_crs(crs)  # 좌표계 통일

# 조건: 두 파일 모두 0.1 이상
mask = (pearson_data >= 0.1) & (spearman_data >= 0.1)
rows, cols = np.where(mask)

# 결과 저장용 리스트
selected_pixels = []

# 각 픽셀 좌표를 검사
for row, col in zip(rows, cols):
    x, y = rasterio.transform.xy(transform, row, col, offset='center')
    point = Point(x, y)
    if gdf.contains(point).any():
        selected_pixels.append((row, col))

# 결과 저장
df = pd.DataFrame(selected_pixels, columns=["row", "col"])
os.makedirs(os.path.dirname(output_csv), exist_ok=True)
df.to_csv(output_csv, index=False, encoding="utf-8-sig")

print(f"✅ 완료: {output_csv}에 {len(df)}개 픽셀 저장됨")


In [ ]:
############# 날짜별 상관계수 값이 지정한 값 이상일 때 픽셀 좌표에서의 상관계수 #############

import os
import pandas as pd
import numpy as np
import rasterio
from scipy.stats import pearsonr, spearmanr
from tqdm import tqdm

# 경로 설정
base_folder = r"E:\projects\smelter\data\python\BlastFurnaces\TAI\0_1"
tif_folder = base_folder  # TIF 파일들
matched_tif_csv = os.path.join(base_folder, "matched_tif.csv")
selected_pixels_csv = os.path.join(base_folder, "correlation_01selected_pixels.csv")  # 수정된 관심 픽셀 파일
activity_csv = r"E:\projects\smelter\data\python\BlastFurnaces\Dillinger_activity_and_production.csv"

# 출력 경로
output_csv = os.path.join(base_folder, "nCorr01positive_vs_activity.csv")  # 바로 base_folder에 저장

# 읽기
df_matched = pd.read_csv(matched_tif_csv)
tif_files = df_matched['filename'].tolist()

df_pixels = pd.read_csv(selected_pixels_csv)
df_pixels.columns = df_pixels.columns.str.lower()
pixel_coords = list(zip(df_pixels['row'], df_pixels['col']))

df_activity = pd.read_csv(activity_csv)
df_activity["Date"] = pd.to_datetime(df_activity["Capture time"])

# 날짜별 n_Corr_positive 계산
records = []

for tif_file in tqdm(tif_files, desc="🔍 날짜별 0 초과 픽셀 개수 계산 중"):
    tif_path = os.path.join(tif_folder, tif_file)
    try:
        with rasterio.open(tif_path) as src:
            img = src.read(1)

            # 관심 있는 픽셀 값 추출
            pixel_values = np.array([img[row, col] for row, col in pixel_coords])

            # 0 초과하는 픽셀 개수 세기
            n_positive = np.sum(pixel_values > 0)

        # 파일명에서 날짜 추출
        parts = tif_file.split('_')
        if len(parts) > 2:
            date_str = parts[2][:8]  # YYYYMMDD
            date = pd.to_datetime(date_str, format="%Y%m%d")
        else:
            continue

        # 해당 날짜에 대응하는 activity 찾기
        activity_row = df_activity[df_activity["Date"] == date]
        if not activity_row.empty:
            activity = activity_row["Activity"].values[0]
            records.append((date, n_positive, activity))

    except Exception as e:
        print(f"⚠️ 오류 발생: {tif_file} - {e}")

# DataFrame 생성
df_result = pd.DataFrame(records, columns=["Date", "n_Corr_positive", "Activity"])
df_result.sort_values("Date", inplace=True)

# 저장
df_result.to_csv(output_csv, index=False, encoding="utf-8-sig")
print(f"✅ 결과 저장 완료: {output_csv}")

# 상관계수 계산
r_pearson, _ = pearsonr(df_result["n_Corr_positive"], df_result["Activity"])
r_spearman, _ = spearmanr(df_result["n_Corr_positive"], df_result["Activity"])

print(f"\n📈 Pearson 상관계수 (n_Corr_positive vs Activity): {r_pearson:.4f}")
print(f"📈 Spearman 상관계수 (n_Corr_positive vs Activity): {r_spearman:.4f}")
